# Entanglement: Bell states and correlations

The Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$
is the simplest entangled state.  The two qubits are correlated:
measuring one instantly determines the other.

In [ ]:
from IPython.display import display
import qiskit as qk
import qiskit_aer as qka

In [ ]:
def show(qc, title):
    print(title)
    print(qc.draw())
    sv = qk.quantum_info.Statevector.from_instruction(
        qc.remove_final_measurements(inplace=False)
    )
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()


def show_mpl(qc, title):
    print(title)
    display(qc.draw(output="mpl"))
    sv = qk.quantum_info.Statevector.from_instruction(
        qc.remove_final_measurements(inplace=False)
    )
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()

## Creating the Bell state $|\Phi^+\rangle$

$H$ puts qubit 0 in superposition.  $CX(0,1)$ entangles it with
qubit 1: when qubit 0 is $|0\rangle$ nothing happens; when it is
$|1\rangle$ qubit 1 gets flipped.

In [ ]:
qc = qk.QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
show_mpl(qc, "|Phi+> = (|00> + |11>)/sqrt(2)")

sv = qk.quantum_info.Statevector.from_instruction(qc)
print("probabilities:", sv.probabilities_dict())

## Perfect correlations

Measuring both qubits always gives the same result: $|00\rangle$ or
$|11\rangle$, each with probability $1/2$.

In [ ]:
qc = qk.QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
backend = qka.AerSimulator()
compiled = qk.transpile(qc, backend)
counts = backend.run(compiled, shots=4000).result().get_counts()
print("counts:", counts)
same = counts.get("00", 0) + counts.get("11", 0)
print(f"fraction same: {same}/4000 = {same/4000:.1%}")
display(qk.visualization.plot_histogram(counts, title="Bell state correlations"))

## Partial measurement collapses the pair

If we measure only qubit 0, qubit 1 is forced into the same state.

In [ ]:
qc = qk.QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure(0, 0)
backend = qka.AerSimulator()
compiled = qk.transpile(qc, backend)
counts = backend.run(compiled, shots=2000).result().get_counts()
print("counts (only qubit 0 measured):", counts)
print("After measuring qubit 0, qubit 1 always matches.")

## Four Bell states

| Name | State | Circuit |
|------|-------|---------|
| $|\Phi^+\rangle$ | $(|00\rangle + |11\rangle)/\sqrt{2}$ | $H$, $CX$ |
| $|\Phi^-\rangle$ | $(|00\rangle - |11\rangle)/\sqrt{2}$ | $X$, $H$, $CX$ |
| $|\Psi^+\rangle$ | $(|01\rangle + |10\rangle)/\sqrt{2}$ | $H$, $CX$, $X_1$ |
| $|\Psi^-\rangle$ | $(|01\rangle - |10\rangle)/\sqrt{2}$ | $X$, $H$, $CX$, $X_1$ |

In [ ]:
bell_states = {
    "Phi+": lambda: _bell(0, False, False),
    "Phi-": lambda: _bell(0, True, False),
    "Psi+": lambda: _bell(0, False, True),
    "Psi-": lambda: _bell(0, True, True),
}

for name, make in bell_states.items():
    qc = make()
    sv = qk.quantum_info.Statevector.from_instruction(qc)
    print(f"|{name}>")
    for b, a in sv.to_dict().items():
        if abs(a) > 1e-10:
            print(f"  {a.real:+.3f} |{b}>")
    print()


def _bell(q0, phase, swap):
    """Build one of the four Bell states."""
    qc = qk.QuantumCircuit(2)
    if phase:
        qc.x(0)
    qc.h(0)
    qc.cx(0, 1)
    if swap:
        qc.x(1)
    return qc

## Summary

- $H + CX$ is the standard Bell-state recipe.
- Bell states exhibit **perfect correlation** (or anti-correlation).
- Measuring one qubit of an entangled pair collapses both.
- The four Bell states span all combinations of $\Phi/\Psi$ and $\pm$.